In [0]:
import random
from datetime import datetime, timedelta

# 1. Configuration
start_date = datetime.now() - timedelta(days=365)
current_time = start_date
# Define the FOLDER path, not a specific file
output_folder = "abfss://battery-data@batteryhealthdatalake.dfs.core.windows.net/bronze/bms_logs/"

tech_fragments = [
    "CELL_VOLT_DIFF: {diff}V across [01, 04, 08]",
    "INT_RES_RISE: {res}mOhm detected on Module_B",
    "THERMAL_GRADIENT: {t_grad}C/s exceeding nominal drift",
    "V_DROOP: Transient sag {sag}V detected"
]

# 2. Generate and Write 52 Files
for week in range(52):
    filename = f"bms_historical_week_{week}.txt"
    full_path = output_folder + filename
    
    # Build the string content for the whole week
    log_content = f"--- BMS HISTORICAL LOG WEEK {week} ---\n"
    for _ in range(50):
        random_offset = random.randint(0, 7 * 24 * 3600) 
        log_time = current_time + timedelta(seconds=random_offset)
        
        log_line = random.choice(tech_fragments).format(
            diff=round(random.uniform(0.1, 0.5), 3),
            res=round(random.uniform(5.0, 15.0), 1),
            t_grad=round(random.uniform(0.5, 2.0), 2),
            sag=round(random.uniform(0.3, 1.2), 2)
        )
        log_content += f"[{log_time.strftime('%Y-%m-%d %H:%M:%S')}] {log_line}\n"
    
    # CRITICAL: Use dbutils to write to Cloud Storage
    dbutils.fs.put(full_path, log_content, overwrite=True)
    
    current_time += timedelta(days=7)

print(f"Successfully wrote 52 files to Azure: {output_folder}")